# Fun

Purpose of this notebook is to actually do training to discover different model 
configurations that would work to train a pretty good TinyGPT. 

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


cuda
NVIDIA A100-SXM4-80GB


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
!git clone https://github.com/sanchitram1/242b-hw3
%cd 242b-hw3

Cloning into '242b-hw3'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 70 (delta 37), reused 55 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 485.94 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/242b-hw3


In [4]:
from tokenizer import build_tokenizer
from training import train_model
import json
from utils import save_json, generate_text
from plot import plot_training_curves, plot_validation_curves
from tokenizer import count_tokens, build_token_memmap
from models import TokenChunkDataset, model_checkpoint_path, load_model
from config import (
    DataConfig,
    GlobalTrainingConfig,
    RunConfig,
    TokenConfig,
    TokenizationConfig,
    ModelConfig,
)
from IPython.display import Markdown, Image

In [5]:
token_config = TokenConfig()
data_config = DataConfig()
global_training_config = GlobalTrainingConfig()
tokenization_config = TokenizationConfig()
run_config = RunConfig("experiments-pt2-vocab=5000")

Created /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt1-100k and subfolders


In [6]:
tokenizer = build_tokenizer(
    tokenization_config, token_config, data_config.training_file_colab
)

Found it in /content/drive/MyDrive/courses/242B/HW3/artifacts/shared/tokenizers/tinystories_bpe_metaspace_3000_1000000.json, loading from there


In [7]:
train_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.training_file_colab
)
valid_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.validation_file_colab
)
print(f"There are {train_token_count} tokens in train, {valid_token_count} valid")

train_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.training_file_colab,
    train_token_count,
)
valid_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.validation_file_colab,
    valid_token_count,
)

There are 189162345 tokens in train, 5192964 valid


In [8]:
train_dataset = TokenChunkDataset(
    train_token_memmap_path, train_token_count, global_training_config.context_length
)
valid_dataset = TokenChunkDataset(
    valid_token_memmap_path, valid_token_count, global_training_config.context_length
)

In [9]:
configs = [
    ModelConfig(
        name="large-plus",
        d_model=384,
        n_heads=6,
        n_layers=6,
        d_ff=1536,
        batch_size=16,
        learning_rate=2.5e-4,
        weight_decay=0.1,
        warmup_steps=1000,
        max_steps=100000,
        use_amp=torch.cuda.is_available(),
    ),
    # ModelConfig(
    #     name="d_model=512",
    #     d_model=512,
    #     n_heads=8,
    #     n_layers=6,
    #     d_ff=2048,
    #     batch_size=64,
    #     learning_rate=2.5e-4,
    #     weight_decay=0.1,
    #     warmup_steps=1000,
    #     max_steps=100000,
    #     use_amp=torch.cuda.is_available(),
    # ),
]


In [10]:
results: dict[str, dict] = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for config in configs:
    metrics_path = run_config.metrics / f"{config.name}.json"
    if metrics_path.exists():
        results[config.name] = json.loads(metrics_path.read_text(encoding="utf-8"))
        continue
    result = train_model(
        run_config,
        global_training_config,
        config,
        tokenizer,
        train_dataset,
        valid_dataset,
        device=device,
    )
    results[config.name] = result
    save_json(result, metrics_path)

    if device.type == "cuda":
        torch.cuda.empty_cache()

plot_training_curves(results, run_config.plots / "training_loss_all_models.png")
plot_validation_curves(results, run_config.plots / "validation_loss_all_models.png")

[half-base] Loss(step=1)=8.04
[half-base] Loss(step=10000)=4.49
[half-base] Loss(step=20000)=3.80
[half-base] Loss(step=30000)=3.52
[half-base] Loss(step=40000)=3.47
[half-base] Loss(step=50000)=3.52
[half-base] Loss(step=60000)=3.52
[half-base] Loss(step=70000)=3.43
[half-base] Loss(step=80000)=3.31
[half-base] Loss(step=90000)=3.24
[half-base] Loss(step=100000)=3.25
[double-base] Loss(step=1)=8.10
[double-base] Loss(step=10000)=3.52
[double-base] Loss(step=20000)=2.90
[double-base] Loss(step=30000)=2.63
[double-base] Loss(step=40000)=2.67
[double-base] Loss(step=50000)=2.50
[double-base] Loss(step=60000)=2.58
[double-base] Loss(step=70000)=2.45
[double-base] Loss(step=80000)=2.48
[double-base] Loss(step=90000)=2.32
[double-base] Loss(step=100000)=2.52
[deep-base] Loss(step=1)=8.04
[deep-base] Loss(step=10000)=3.68
[deep-base] Loss(step=20000)=2.96
[deep-base] Loss(step=30000)=2.91
[deep-base] Loss(step=40000)=2.74
[deep-base] Loss(step=50000)=2.76
[deep-base] Loss(step=60000)=2.58
[d

In [11]:
SAMPLE_PROMPTS = (
    "Once upon a time",
    "A little girl and her pet cat",
    "A fairy woke up",
    "A lion, a bear, and a penguin",
    "The two children",
)

In [12]:
generations: list[dict] = []
output_settings = [
    {"temperature": 0.7, "top_k": 30},  # balanced
    {"temperature": 0.5, "top_k": 10},  # conservative
    {"temperature": 0.8, "top_k": 50},  # experimental
]

for prompt in SAMPLE_PROMPTS:
    for config in configs:
        model_path = model_checkpoint_path(run_config, config)
        model = load_model(model_path, device)

        for setting in output_settings:
            temperature = setting["temperature"]
            top_k = setting["top_k"]
            generation = generate_text(
                token_config,
                global_training_config,
                model,
                tokenizer,
                prompt,
                device,
                120,
                temperature,
                top_k,
            )

            generations.append(
                {
                    "model": config.name,
                    "prompt": prompt,
                    "generated_text": generation,
                    "temperature": temperature,
                    "top_k": top_k,
                }
            )

samples_dir = run_config.run_dir / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

generations_path = samples_dir / "generations.json"
generations_path.write_text(
    json.dumps(generations, indent=2),
    encoding="utf-8",
)

print(f"Saved generations to {generations_path}")


Saved generations to /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt1-100k/samples/generations.json


In [13]:
from dataclasses import asdict

manifest = {
    "run_id": run_config.run_id,
    "run_dir": str(run_config.run_dir),
    "device": str(device),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data": {
        "training_file": str(data_config.training_file),
        "validation_file": str(data_config.validation_file),
    },
    "tokenization": asdict(tokenization_config),
    "global_training": asdict(global_training_config),
    "models": [asdict(config) for config in configs],
    "artifacts": {
        "models_dir": str(run_config.models),
        "metrics_dir": str(run_config.metrics),
        "plots_dir": str(run_config.plots),
        "samples_dir": str(samples_dir),
        "generations": str(generations_path),
    },
    "generation_settings": output_settings,
}

manifest_path = run_config.run_dir / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(f"Saved manifest to {manifest_path}")


Saved manifest to /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt1-100k/manifest.json


In [35]:
for sample in generations:
    name = sample["model"]
    display(
        Markdown(
            f"**{name}**: Prompt=`{sample['prompt']}` *({sample['temperature']}, {sample['top_k']})*"
        )
    )
    display(Markdown(sample["generated_text"]))


**small**: Prompt=`Once upon a time` *(0.7, 30)*

Once upon a time there was a big tree. She loved to climb the tree and make pretty leaves. One day, the tree was flying high in the sky. It was a big tree. The tree was very hot in the tree. The tree was sad and hungry. The tree was not scared anymore. It was a good tree!

**small**: Prompt=`Once upon a time` *(0.5, 10)*

Once upon a time there was a little girl named Lily. Lily had a big toy box that she loved to play with her ball. One day, she went to the park with her mom. She saw a big box of toys. Lily wanted to play with the box for her toe. She took the box and put it back to her room. Lily was very happy. She said, "Thank you, Lily!" Lily was happy and played with her toys all day.

**small**: Prompt=`Once upon a time` *(0.8, 50)*

...

A little girl and her pet cat named Lily, went for a walk. They saw a big tree with many leaves. Max was very happy to have a friend to play. Lily said, "Let's see that a little bird in the tree!" So, they started to play catch with the bird in the sun. They were very happy and full. The bird knew that Max was right. Max and Lily played with the bird. They were very good friends and had lots of fun.

**large**: Prompt=`A little girl and her pet cat` *(0.5, 10)*

A little girl and her pet cat were very happy. They played together all day long, and the little girl was not scared anymore.

**large**: Prompt=`A little girl and her pet cat` *(0.8, 50)*

A little girl and her pet cat were shocked. They looked at each other and decided to get her ball back. Amy said: "Let's get the ball down and watch the ball go away. Maybe it is too big and can get it out."
They tried to grab the ball, but it was too hard for Amy and her best. They were too afraid. Then, Amy started to cry and ran around the house. She saw the ball near the door and knew how much it was. She tried to get it, but it was too late. The ball was too big for her to get out. Amy was sad and scared. She tried to help, but the ball did not move.